In [19]:
"""
이름: 왜용
목적: 어린 아이들의 순수한 질문을 학습할 수 있게 변환
핵심 기능: 
  1. 질문을 과학적 해석으로 풀어서 눈높이 설명
  2. 간단한 실험이 가능하면 추천
  3. 학습 기록과 관련 질문 생성

그래프 구조:
[질문 입력]
    ↓
[질문 의도 분석]
    ↓
[학습 주제 변환]
    ↓
[연령/수준 판단]
    ↓
[눈높이 설명 생성]
    ↓
[이해도 확인 질문]
    ↓
[아이 답변 평가]
    ↓
 ┌───────────────┬───────────────┐
 ↓               ↓               ↓
[재설명]      [퀴즈 생성]      [실험 추천]
 ↓               ↓               ↓
[학습 기록 저장 및 관련 질문 추천]
"""

'\n이름: 왜용\n목적: 어린 아이들의 순수한 질문을 학습할 수 있게 변환\n핵심 기능: \n  1. 질문을 과학적 해석으로 풀어서 눈높이 설명\n  2. 간단한 실험이 가능하면 추천\n  3. 학습 기록과 관련 질문 생성\n\n그래프 구조:\n[질문 입력]\n    ↓\n[질문 의도 분석]\n    ↓\n[학습 주제 변환]\n    ↓\n[연령/수준 판단]\n    ↓\n[눈높이 설명 생성]\n    ↓\n[이해도 확인 질문]\n    ↓\n[아이 답변 평가]\n    ↓\n ┌───────────────┬───────────────┐\n ↓               ↓               ↓\n[재설명]      [퀴즈 생성]      [실험 추천]\n ↓               ↓               ↓\n[학습 기록 저장 및 관련 질문 추천]\n'

In [20]:
import json
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, List
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

llm = init_chat_model(model="openai:gpt-4o-mini")

In [21]:
class QuestionPurpose(BaseModel):
    intent_type: str = Field(description="질문 의도 유형")
    child_curiosity: str = Field(description="아이가 궁금해하는 핵심 내용")
    learning_need: str = Field(description="학습에 필요한 개념")
    emotion: str = Field(description="질문에 담긴 감정")

class StudySubject(BaseModel):
    title: str = Field(description="학습 주제 제목")
    core_concept: str = Field(description="핵심 개념")
    learning_goal: str = Field(description="학습 목표")
    keywords: List[str] = Field(description="관련 키워드 목록")

class State(TypedDict):
    question: str
    question_purpose: QuestionPurpose
    study_subject: StudySubject
    pass

graph_builder = StateGraph(State)


In [22]:
def question_purpose_analysis(state: State):
    
    print(f"question_purpose_analysis => {state['question']}")
    response = llm.with_structured_output(QuestionPurpose).invoke(f"""
    당신은 어린이의 순수한 질문을 학습으로 연결하는 교육 에이전트입니다.

    아래 질문의 목적을 분석해주세요.

    질문:
    {state["question"]}
    """)
    return {
        "question_purpose": response
    }

def convert_study_subject(state: State):
    print(f"convert_study_subject => {state['question_purpose']}")
    question_purpose = state["question_purpose"]
    response = llm.with_structured_output(StudySubject).invoke(f"""
    당신은 어린이의 질문을 학습 주제로 변환하는 교육 에이전트입니다.

    원래 질문:
    {state["question"]}

    질문 목적:
    - 의도 유형: {question_purpose.intent_type}
    - 아이의 궁금증: {question_purpose.child_curiosity}
    - 학습 필요 개념: {question_purpose.learning_need}
    - 감정: {question_purpose.emotion}

    위 내용을 바탕으로 어린이가 배울 수 있는 학습 주제를 선정해주세요.
    """)

    return {
        "study_subject": response
    }

In [23]:
graph_builder.add_node('question_purpose_analysis', question_purpose_analysis)
graph_builder.add_node('convert_study_subject', convert_study_subject)

graph_builder.add_edge(START, 'question_purpose_analysis')
graph_builder.add_edge("question_purpose_analysis", 'convert_study_subject')
graph_builder.add_edge("convert_study_subject", END)

graph = graph_builder.compile()

In [24]:
graph.invoke({ "question": "하늘은 왜 파래요?"})

question_purpose_analysis => 하늘은 왜 파래요?
convert_study_subject => intent_type='질문에 대한 호기심을 해소하기 위한 것' child_curiosity='자연현상에 대한 순수한 관심과 궁금증' learning_need='하늘의 색깔 변화와 그 원인에 대한 과학적 이해' emotion='호기심과 흥미가 가득한 상태'


{'question': '하늘은 왜 파래요?',
 'question_purpose': QuestionPurpose(intent_type='질문에 대한 호기심을 해소하기 위한 것', child_curiosity='자연현상에 대한 순수한 관심과 궁금증', learning_need='하늘의 색깔 변화와 그 원인에 대한 과학적 이해', emotion='호기심과 흥미가 가득한 상태'),
 'study_subject': StudySubject(title='하늘의 색깔과 자연 현상', core_concept='하늘의 색은 대기와 태양의 빛에 의해 결정된다.', learning_goal='하늘이 어떤 원리로 파랗게 보이는지를 이해하고, 다른 날씨나 시간대에서 하늘의 색이 어떻게 변하는지를 배운다.', keywords=['하늘의 색깔', '파란 하늘', '대기', '태양빛', '자연현상', '구름의 색', '일출과 일몰'])}